# Combined tasmax + tasmin Indices (DTR, ETR, vDTR)

Some indices need **both** `tasmax` and `tasmin` together (they're not computable from
either variable alone): the Diurnal Temperature Range (**DTR**), Extreme Temperature
Range (**ETR**), and the day-to-day variability of DTR (**vDTR**).

**Assumptions:**
- Single `.nc` file each for `tasmax` and `tasmin`, covering the same period
- Units already in `degC`


In [ ]:
# !pip install icclim xarray netCDF4 matplotlib pandas --quiet

In [ ]:
import os
import warnings
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import icclim

warnings.filterwarnings("ignore")


In [ ]:
# ============================================================
# CONFIGURATION — EDIT THESE PATHS FOR YOUR DATA
# ============================================================
TASMAX_FILE = "/path/to/your/tasmax_daily.nc"
TASMIN_FILE = "/path/to/your/tasmin_daily.nc"
OUT_DIR = "./outputs/combined_tx_tn"
os.makedirs(OUT_DIR, exist_ok=True)

TIME_RANGE = None
SLICE_MODE = "year"


## Compute DTR, ETR, vDTR

In [ ]:
INDICES = ["DTR", "ETR", "vDTR"]

results = {}

for idx_name in INDICES:
    print(f"Computing {idx_name} ...")
    out_file = os.path.join(OUT_DIR, f"{idx_name}.nc")
    try:
        ds_out = icclim.index(
            index_name=idx_name,
            in_files={"tasmax": {"study": TASMAX_FILE}, "tasmin": {"study": TASMIN_FILE}},
            slice_mode=SLICE_MODE,
            time_range=TIME_RANGE,
            out_file=out_file,
        )
        results[idx_name] = ds_out
        print(f"  -> saved to {out_file}")
    except Exception as e:
        print(f"  !! FAILED for {idx_name}: {e}")
        print("  If this fails, check icclim's docs for the exact multi-variable ")
        print("  'in_files' dict format for your installed icclim version.")


## Quick-look plots

In [ ]:
for idx_name, ds_out in results.items():
    data_vars = [v for v in ds_out.data_vars if not v.endswith("_thresholds")]
    if not data_vars:
        continue
    da = ds_out[data_vars[0]]
    spatial_dims = [d for d in da.dims if d not in ("time",)]
    ts = da.mean(dim=spatial_dims, skipna=True) if spatial_dims else da

    plt.figure(figsize=(7, 3))
    ts.plot()
    plt.title(f"{idx_name} — spatial mean")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{idx_name}_timeseries.png"), dpi=120)
    plt.show()
